In [ ]:
# https://michong00.github.io/assets/projects/snake-blanks.ipynb
# or https://michong00.github.io/teaching

In [1]:
pip install pygame

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pygame
import time
import random

pygame.init()

WHITE = (255, 255, 255)
YELLOW = (255, 255, 102)
BLACK = (0, 0, 0)
RED = (213, 50, 80)
GREEN = (0, 255, 0)
BLUE = (50, 153, 213)

DIS_WIDTH = 600
DIS_HEIGHT = 400

dis = pygame.display.set_mode((DIS_WIDTH, DIS_HEIGHT))
pygame.display.set_caption('Snake Game')

clock = pygame.time.Clock()

snake_block = 10
snake_speed = 15

font_style = pygame.font.SysFont("bahnschrift", 25)
score_font = pygame.font.SysFont("comicsansms", 35)

def calc_score(score):
    value = score_font.render(f"Your score: {score}", True, YELLOW)
    dis.blit(value, [0,0])

def our_snake(snake_block, snake_list):
    for x in snake_list: # snake_list = [[1,3], [1,4]]
        pygame.draw.rect(dis, BLACK, [x[0], x[1], snake_block, snake_block])

def message(msg, color):
    mesg = font_style.render(msg, True, color)
    dis.blit(mesg, [DIS_WIDTH / 6, DIS_HEIGHT / 3])

def game_loop():
    game_over = False # flag variable
    game_close = False
    # start coordinates for the snake (centre)
    x1 = DIS_WIDTH / 2
    y1 = DIS_HEIGHT / 2
    
    x1_change = 0
    y1_change = 0
    snake_list = []
    snake_length = 1
    # coordinates for food
    foodx = round(random.randrange(0, DIS_WIDTH - snake_block) / 10.0) * 10.0
    foody = round(random.randrange(0, DIS_HEIGHT - snake_block) / 10.0) * 10.0

    while not game_over:
        while game_close:
            # when you lose
            dis.fill(BLUE)
            message("You lost! :( Press 'C' to play again or 'Q' to quit.", RED)
            calc_score(snake_length - 1)
            pygame.display.update()

            for event in pygame.event.get():
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_q:
                        game_close = False
                        game_over = True
                    if event.key == pygame.K_c:
                        game_loop()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                game_over = True
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_LEFT:
                    x1_change = -snake_block
                    y1_change = 0
                elif event.key == pygame.K_RIGHT:
                    x1_change = snake_block
                    y1_change = 0
                elif event.key == pygame.K_UP:
                    x1_change = 0
                    y1_change = -snake_block
                elif event.key == pygame.K_DOWN:
                    x1_change = 0
                    y1_change = snake_block
        # if you run into a wall
        if x1 >= DIS_WIDTH or x1 < 0 or y1 >= DIS_HEIGHT or y1 <0:
            game_close = True
        # update screen
        x1 += x1_change # x1 = x1 + x1_change
        y1 += y1_change
        dis.fill(BLUE)
        pygame.draw.rect(dis, GREEN, [foodx, foody, snake_block, snake_block])
        
        snake_head = [x1, y1]

        snake_list.append(snake_head)
        if len(snake_list) > snake_length:
            del snake_list[0]
        # check if you ate yourself
        for x in snake_list[:-1]:
            if x == snake_head:
                game_close = True
        # update display
        our_snake(snake_block, snake_list)
        calc_score(snake_length - 1)
        pygame.display.update()

        # check if you ate the food, generate new food
        if x1 == foodx and y1 == foody:
            foodx = round(random.randrange(0, DIS_WIDTH - snake_block) / 10.0) * 10.0
            foody = round(random.randrange(0, DIS_HEIGHT - snake_block) / 10.0) * 10.0
            snake_length += 1
        clock.tick(snake_speed)
    pygame.quit()
    quit()

game_loop()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


: 

# Exercises
1. Change colors of background / snake / text / food (at least one).
2. Change the size of the grid to have a bigger or smaller window.
3. Let user use WASD movement, instead of arrow keys.

In [ ]:
# Rafa's version
# basic loop & draw grid
# snake movement
# food
import random

import pygame

# --- Setup ---
pygame.init()

# Designing a grid on top of pixel coordinates
CELL_SIZE = 20  # pixels
GRID_WIDTH = 30  # cells across
GRID_HEIGHT = 20  # cells down

WINDOW_WIDTH = GRID_WIDTH * CELL_SIZE
WINDOW_HEIGHT = GRID_HEIGHT * CELL_SIZE

screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
pygame.display.set_caption("Snake")

clock = pygame.time.Clock()
FPS = 60

BACKGROUND_COLOR = (10, 10, 10)
GRID_LINE_COLOR = (40, 40, 40)
SNAKE_COLOR = (0, 200, 0)
HEAD_COLOR = (0, 255, 100)
FOOD_COLOR = (220, 60, 60)
TEXT_COLOR = (255, 255, 255)

font = pygame.font.SysFont(None, 28)
big_font = pygame.font.SysFont(None, 48)

STARTING_MOVE_INTERVAL = 150  # milliseconds per grid step at the start of a run
MIN_MOVE_INTERVAL = 60  # the fastest the snake is ever allowed to move
SPEEDUP_PER_FOOD = 5  # how many ms faster the snake gets per food eaten


def draw_grid(surface):
    """Draw faint grid lines so each cell is visible."""
    for col in range(GRID_WIDTH + 1):
        x = col * CELL_SIZE
        pygame.draw.line(surface, GRID_LINE_COLOR, (x, 0), (x, WINDOW_HEIGHT))

    for row in range(GRID_HEIGHT + 1):
        y = row * CELL_SIZE
        pygame.draw.line(surface, GRID_LINE_COLOR, (0, y), (WINDOW_WIDTH, y))


def random_empty_cell(occupied_cells):
    """Pick a random grid cell that isn't currently occupied by the snake."""
    while True:
        cell = (random.randint(0, GRID_WIDTH - 1), random.randint(0, GRID_HEIGHT - 1))
        if cell not in occupied_cells:
            return cell


def hits_wall(head):
    """True if the head has gone outside the grid's valid range."""
    x, y = head
    return x < 0 or x >= GRID_WIDTH or y < 0 or y >= GRID_HEIGHT


def hits_self(snake_body):
    """True if the head's position already appears elsewhere in the body."""
    return snake_body[0] in snake_body[1:]


def draw_centered_text(surface, text, font_obj, color, y_offset=0):
    """Render text centered horizontally (and vertically, plus an offset)."""
    text_surface = font_obj.render(text, True, color)
    rect = text_surface.get_rect(
        center=(WINDOW_WIDTH // 2, WINDOW_HEIGHT // 2 + y_offset)
    )
    surface.blit(text_surface, rect)


# Game state variables
game_state = "menu"  # Three values: "menu", "playing", "game_over".

snake = []
direction = (1, 0)
score = 0
food = (0, 0)
move_timer = 0
move_interval = STARTING_MOVE_INTERVAL


def reset_game():
    """Reinitialise every variable that changes during play."""
    global snake, direction, score, food, move_timer, move_interval

    snake = [(10, 10), (9, 10), (8, 10)]
    direction = (1, 0)
    score = 0
    food = random_empty_cell(snake)
    move_timer = 0
    move_interval = STARTING_MOVE_INTERVAL


running = True

while running:
    # --- 1. Handle events ---
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        elif event.type == pygame.KEYDOWN:
            if game_state == "menu":
                if event.key == pygame.K_SPACE:
                    reset_game()
                    game_state = "playing"

            elif game_state == "playing":
                if event.key == pygame.K_UP and direction != (0, 1):
                    direction = (0, -1)
                elif event.key == pygame.K_DOWN and direction != (0, -1):
                    direction = (0, 1)
                elif event.key == pygame.K_LEFT and direction != (1, 0):
                    direction = (-1, 0)
                elif event.key == pygame.K_RIGHT and direction != (-1, 0):
                    direction = (1, 0)

            elif game_state == "game_over" and event.key == pygame.K_r:
                reset_game()
                game_state = "playing"

    # --- 2. Update state ---
    if game_state == "playing":
        move_timer += clock.get_time()

        if move_timer >= move_interval:
            move_timer = 0

            # Moving the whole snake on a fixed tick
            head_x, head_y = snake[0]
            new_head = (head_x + direction[0], head_y + direction[1])
            snake.insert(0, new_head)

            # Checking collisions in a sensible order: food first.
            if new_head == food:
                score += 1
                food = random_empty_cell(snake)
                move_interval = max(MIN_MOVE_INTERVAL, move_interval - SPEEDUP_PER_FOOD)
            else:
                snake.pop()

            if hits_wall(new_head) or hits_self(snake):
                game_state = "game_over"

    # --- 3. Draw the frame ---
    screen.fill(BACKGROUND_COLOR)
    draw_grid(screen)

    if game_state == "menu":
        draw_centered_text(screen, "SNAKE", big_font, TEXT_COLOR, y_offset=-30)
        draw_centered_text(
            screen, "Press SPACE to start", font, TEXT_COLOR, y_offset=20
        )

    elif game_state == "playing":
        food_rect = (food[0] * CELL_SIZE, food[1] * CELL_SIZE, CELL_SIZE, CELL_SIZE)
        pygame.draw.rect(screen, FOOD_COLOR, food_rect)

        for index, segment in enumerate(snake):
            seg_x, seg_y = segment
            rect = (seg_x * CELL_SIZE, seg_y * CELL_SIZE, CELL_SIZE, CELL_SIZE)
            color = HEAD_COLOR if index == 0 else SNAKE_COLOR
            pygame.draw.rect(screen, color, rect)

        score_surface = font.render(f"Score: {score}", True, TEXT_COLOR)
        screen.blit(score_surface, (10, 10))

    elif game_state == "game_over":
        for index, segment in enumerate(snake):
            seg_x, seg_y = segment
            rect = (seg_x * CELL_SIZE, seg_y * CELL_SIZE, CELL_SIZE, CELL_SIZE)
            color = HEAD_COLOR if index == 0 else SNAKE_COLOR
            pygame.draw.rect(screen, color, rect)

        draw_centered_text(screen, "GAME OVER", big_font, TEXT_COLOR, y_offset=-30)
        draw_centered_text(
            screen,
            f"Score: {score}  --  Press R to restart",
            font,
            TEXT_COLOR,
            y_offset=20,
        )

    pygame.display.flip()
    clock.tick(FPS)


pygame.quit()